# 🇹🇭 LLM Thai News Translation (Optimized Prompt)

**Prompt Engineering:**
- Clear structure with Thai instructions
- Few-shot example for format
- Prioritized rules
- Character limits instead of word limits

In [ ]:
# CELL 1: IMPORTS & CONFIG
import pandas as pd
from datetime import datetime

from llama_cpp import Llama
print("✅ llama-cpp-python loaded")

class CFG:
    INPUT_CSV = "scraped_news_v10_full.csv"
    MODEL_PATH = "../data/models/typhoon2.5-qwen3-4b-q4_k_m.gguf"
    
    TEST_LIMIT = 1
    
    N_CTX = 8192
    N_GPU_LAYERS = -1
    TEMPERATURE = 0.4
    MAX_TOKENS = 2500

print(f"📂 Input: {CFG.INPUT_CSV}")

In [ ]:
# CELL 2: LOAD MODEL
print("Loading Typhoon 2.5...")

llm = Llama(
    model_path=CFG.MODEL_PATH,
    n_ctx=CFG.N_CTX,
    n_gpu_layers=CFG.N_GPU_LAYERS,
    verbose=False
)

print("✅ Model loaded!")

In [ ]:
# CELL 3: LOAD FIRST ARTICLE
df = pd.read_csv(CFG.INPUT_CSV)
article = df.iloc[0]

title = article.get('headline', '')
content = article.get('full_content', '') or article.get('content_snippet', '')
source = article.get('source', '')
url = article.get('url', '')
published = article.get('published', '')

print(f"📰 Article: {title[:80]}...")
print(f"📍 Source: {source}")
print(f"📝 Content: {len(content)} chars")

In [ ]:
# CELL 4: OPTIMIZED PROMPT (Thai-first, with example)
PROMPT = f'''คุณคือนักแปลข่าวมืออาชีพ แปลข่าวเป็นภาษาไทยสำหรับผู้อ่านทั่วไปที่สนใจเทคโนโลยี

## ข่าวต้นฉบับ
หัวข้อ: {title}
แหล่งที่มา: {source}
เนื้อหา: {content[:2000]}

## กฎสำคัญ
1. ชื่อคน: เขียนไทยตามด้วยอังกฤษในวงเล็บ เช่น "อีลอน มัสก์ (Elon Musk)"
2. คำพูด: เก็บต้นฉบับอังกฤษและแปลไทยในวงเล็บ เช่น "Move fast" (เดินหน้าอย่างรวดเร็ว)
3. ตัวย่อ: เขียนชื่อเต็มก่อน เช่น ปัญญาประดิษฐ์ (AI)
4. โทน: เป็นกลาง เป็นทางการ สำหรับห้องข่าว
5. เข้าใจง่าย: อธิบายศัพท์เทคนิคให้เด็กเข้าใจได้

## ตัวอย่างรูปแบบ
---
คำสำคัญ: AI, startup, funding | เอไอ, สตาร์ทอัพ, ระดมทุน
---
พาดหัว: สตาร์ทอัพเอไอระดมทุน 50 ล้านดอลลาร์ ขยายธุรกิจในเอเชีย
---
นำเรื่อง: บริษัท XYZ สตาร์ทอัพปัญญาประดิษฐ์ (AI) จากซิลิคอนแวลลีย์ ประกาศปิดรอบระดมทุน 50 ล้านดอลลาร์สหรัฐ เมื่อวันที่ 15 ธันวาคม โดยมีแผนขยายธุรกิจสู่ตลาดเอเชีย
---
เนื้อหา:
[ย่อหน้าที่ 1: บริบทและความสำคัญของข่าว]

[ย่อหน้าที่ 2: ข้อเท็จจริงและตัวเลขสำคัญ]

[ย่อหน้าที่ 3: คำพูดจากผู้เกี่ยวข้อง พร้อมต้นฉบับอังกฤษ]

[ย่อหน้าที่ 4: ผลกระทบและความหมาย]
---
วิเคราะห์:
[วิเคราะห์ว่าข่าวนี้เกี่ยวข้องกับประเทศไทยอย่างไร]
[โอกาสทางเศรษฐกิจหรือสังคมสำหรับคนไทย]
[ตัวอย่างการนำไปใช้ประโยชน์]
---
ที่มา: {source} — {url}
---

## เริ่มแปล (ใช้ --- คั่นแต่ละส่วน)

คำสำคัญ:'''

print(f"📝 Prompt length: {len(PROMPT)} chars")
print("✅ Optimized prompt ready")

In [ ]:
# CELL 5: GENERATE TRANSLATION
import time

print("🔄 Generating translation...")
print("⏳ Expected: 2-3 minutes...")

start = time.time()

response = llm(
    PROMPT,
    max_tokens=CFG.MAX_TOKENS,
    temperature=CFG.TEMPERATURE,
    stop=["## จบ", "\n\n\n\n"],
    echo=False
)

output = response['choices'][0]['text'].strip()
elapsed = time.time() - start

print(f"\n✅ Done in {elapsed:.1f} seconds")
print(f"📊 Output length: {len(output)} chars")

In [ ]:
# CELL 6: DISPLAY FULL OUTPUT
print("="*80)
print("📰 TRANSLATED ARTICLE")
print("="*80)
print(f"\nOriginal: {title}\n")
print("---")
print(output)

In [ ]:
# CELL 7: PARSE AND DISPLAY SECTIONS
sections = output.split('---')

section_names = ['คำสำคัญ', 'พาดหัว', 'นำเรื่อง', 'เนื้อหา', 'วิเคราะห์', 'ที่มา']

print("\n" + "="*80)
print("📋 PARSED SECTIONS")
print("="*80)

for i, section in enumerate(sections):
    section = section.strip()
    if section:
        name = section_names[min(i, len(section_names)-1)] if i < len(section_names) else f'ส่วนที่ {i+1}'
        
        print(f"\n{'─'*40}")
        print(f"📌 {name}")
        print(f"{'─'*40}")
        
        lines = section.split('\n')
        for line in lines:
            if line.strip():
                print(line.strip())

In [ ]:
# CELL 8: QUALITY CHECK
print("\n" + "="*80)
print("📊 QUALITY CHECK")
print("="*80)

for section in sections:
    if 'พาดหัว' in section or section.strip().startswith('พาดหัว'):
        headline = section.replace('พาดหัว:', '').replace('พาดหัว', '').strip()
        headline = headline.split('\n')[0]
        print(f"\n✅ พาดหัว: {headline}")
        print(f"   ความยาว: {len(headline)} ตัวอักษร (จำกัด 70)")
        if len(headline) > 70:
            print("   ⚠️ เกินความยาว!")
        break

thai_chars = sum(1 for c in output if '\u0E00' <= c <= '\u0E7F')
print(f"\n📊 Thai characters: {thai_chars}")
print(f"📊 Total characters: {len(output)}")
print(f"📊 Thai ratio: {thai_chars/len(output)*100:.1f}%")

In [23]:
# CELL 9: SAVE OUTPUT AS TXT
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"translated_{timestamp}.txt"

full_content = f"""================================================================================
ORIGINAL ARTICLE
================================================================================
Title: {title}
Source: {source}
URL: {url}
Published: {published}

================================================================================
THAI TRANSLATION
================================================================================
{output}

================================================================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================================
"""

with open(filename, 'w', encoding='utf-8') as f:
    f.write(full_content)

print(f"💾 Saved to: {filename}")
print(f"📊 File size: {len(full_content)} chars")

💾 Saved to: translated_20251217_032819.txt
📊 File size: 5667 chars
